# 50 dictionary declarations
Run the declarations, then the related cases. Expected refusals are caught and checked.

In [1]:
"""Validate dictionary declarations and their supported boundaries.
Run: .venv/bin/python examples/step_20_dictionaries/50_dictionary_declarations.py
"""

from typing import Annotated

from pydantic import Field

from aoa.action_machine.domain import AssociationOne, BaseCoreEntity, BaseEnumEntity, NoInverse, NoRelation, value
from aoa.action_machine.domain.base_domain import BaseDomain
from aoa.action_machine.domain.exceptions import EntityFieldDeclarationError
from aoa.action_machine.graph.validators.entity_field_validator import validate_entity_fields
from aoa.action_machine.intents.entity import entity


class Shop50Domain(BaseDomain):
    """Group the shop declarations."""

    name = "shop"
    description = "Shop model"


@entity(description="Regions the shop sells in", domain=Shop50Domain)
class Region50EnumEntity(BaseEnumEntity):
    """Declare the region vocabulary."""

    EU = value(code="eu", display_name="Europe")
    US = value(code="us", display_name="United States")


@entity(description="Offer", domain=Shop50Domain)
class Offer50CoreEntity(BaseCoreEntity):
    """Try to serve several regions from one field."""

    id: str = Field(description="Offer identifier")
    regions: tuple[Region50EnumEntity, ...] = Field(description="Regions served")


def case_50() -> None:
    """Run this one learning experiment."""
    try:
        validate_entity_fields([Offer50CoreEntity])
    except EntityFieldDeclarationError as error:
        print(type(error).__name__ + ":", error)
    else:
        raise AssertionError("Expected declaration refusal")


class Shop51Domain(BaseDomain):
    """Group the shop declarations."""

    name = "shop"
    description = "Shop model"


@entity(description="Regions the shop sells in", domain=Shop51Domain)
class Region51EnumEntity(BaseEnumEntity):
    """Declare the region vocabulary."""

    EU = value(code="eu", display_name="Europe")


@entity(description="Offer", domain=Shop51Domain)
class Offer51CoreEntity(BaseCoreEntity):
    """Wrap a shared value in an association."""

    id: str = Field(description="Offer identifier")
    region: Annotated[AssociationOne[Region51EnumEntity], NoInverse()] = Field(description="Primary region")


def case_51() -> None:
    """Run this one learning experiment."""
    try:
        validate_entity_fields([Offer51CoreEntity])
    except EntityFieldDeclarationError as error:
        print(type(error).__name__ + ":", error)
    else:
        raise AssertionError("Expected declaration refusal")


class Shop52Domain(BaseDomain):
    """Group the shop declarations."""

    name = "shop"
    description = "Shop model"


@entity(description="Regions the shop sells in", domain=Shop52Domain)
class Region52EnumEntity(BaseEnumEntity):
    """Declare the region vocabulary."""

    EU = value(code="eu", display_name="Europe")


@entity(description="Offer", domain=Shop52Domain)
class Offer52CoreEntity(BaseCoreEntity):
    """Mark a reference as an embedded value."""

    id: str = Field(description="Offer identifier")
    region: Annotated[Region52EnumEntity, NoRelation("a value")] = Field(description="Primary region")


def case_52() -> None:
    """Run this one learning experiment."""
    try:
        validate_entity_fields([Offer52CoreEntity])
    except EntityFieldDeclarationError as error:
        print(type(error).__name__ + ":", error)
    else:
        raise AssertionError("Expected declaration refusal")


def main() -> None:
    """Run the related declaration cases."""
    case_50()
    case_51()
    case_52()




In [2]:
main()


EntityFieldDeclarationError: Field 'regions' on entity 'Offer50CoreEntity' is invalid: a collection of dictionary values has no declared form (tuple[__main__.Region50EnumEntity, ...]); a dictionary reference is one value. Write Region50EnumEntity, or Region50EnumEntity | None for an optional one.
EntityFieldDeclarationError: Field 'region' on entity 'Offer51CoreEntity' is invalid: the field's type is a relation container around a dictionary (Region51EnumEntity); a dictionary value is a shared reference, not an association target. Write Region51EnumEntity on the field instead.
EntityFieldDeclarationError: Field 'region' on entity 'Offer52CoreEntity' is invalid: NoRelation marks an embedded structural entity, and this field's type is a dictionary (Region52EnumEntity): the bare dictionary type already declares the reference. Remove the marker and keep Region52EnumEntity on the field.
